In [49]:
# ============================================================
# CELL 1 — Imports, Seeds, Paths, Config
# ============================================================

import os, json, time, copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ── Paths ────────────────────────────────────────────────────
# VeReMi dataset location on Kaggle
DATA_DIR    = '/kaggle/input/datasets/ivarprudnikov/veremi-extension-data-1-21-gb'
# Where YOUR trained results are saved
OUTPUT_DIR  = '/kaggle/working'
# Where a teammate's pre-trained results might be uploaded
# Teammate should upload as a Kaggle dataset named 'veremi-results'
UPLOAD_DIR  = '/kaggle/input/datasets/prasannampk/veremi-results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Experiment Config ─────────────────────────────────────────
CFG = {
    'window_size'   : 10,
    'hidden_size'   : 128,      # Increased from 64 for better feature extraction
    'num_classes'   : 20,       
    'num_layers'    : 2,
    'dropout'       : 0.3,
    'lr'            : 1e-3,
    'local_epochs'  : 3,        
    'batch_size'    : 256,      
    'num_rounds'    : 40,       
    'buffer_size'   : 15,       
    'sparse_frac'   : 0.4,      
    'static_churn'  : 0.3,      # Lower churn means more clients stay active
    'train_split'   : 0.8,
}
print(f"Cell 1 complete — Rounds: {CFG['num_rounds']} | Buffer K: {CFG['buffer_size']}")

Device: cuda
Cell 1 complete — Rounds: 40 | Buffer K: 15


# AFL-VeReMi-Bench: Asynchronous Federated Learning on VeReMi Dataset
**Models:** BiLSTM, GRU | **Strategies:** NoWeighting, RoundGap, CosineSim | **Churn:** Static

---
### How to use this notebook
- **Cells 1–3**: Shared infrastructure — run once before anything else
- **Cells 4–15**: One cell per run (12 total) — each is independent and skip-safe
- **Cells 16–18**: Analysis — load saved results and generate plots

**Checkpoint logic:** Each run cell checks `/kaggle/input/` (uploaded by teammate) AND `/kaggle/working/` (your own output). If found in either place, training is skipped automatically.

### Dataset Classification: Extended VeReMi (20 Classes)

The following table defines the behavior of each class in the dataset. These are grouped into three main categories: **Normal**, **Faults** (sensor errors), and **Attacks** (malicious behavior).

| Category | ID | Class Name | Description |
| :--- | :---: | :--- | :--- |
| **Normal** | 0 | **Normal** | Truthful behavior; broadcasts accurate GPS and Speed data. |
| **Faults** | 1 | Constant Position | Sensor is stuck; broadcasts the same position regardless of movement. |
| **Faults** | 2 | Constant Position Offset | Broadcasts a fixed distance away from the true position. |
| **Faults** | 3 | Random Position | Broadcasts chaotic, random coordinates within a range. |
| **Faults** | 4 | Random Position Offset | True coordinates plus a random noise offset. |
| **Faults** | 5 | Constant Speed | Speedometer is stuck; broadcasts a fixed velocity. |
| **Faults** | 6 | Constant Speed Offset | Broadcasts a fixed speed difference (e.g., always +20km/h). |
| **Faults** | 7 | Random Speed | Broadcasts chaotic, random speed values. |
| **Faults** | 8 | Random Speed Offset | True speed plus a random noise offset. |
| **Attacks** | 9 | Disruptive | Intentional interference with network communication. |
| **Attacks** | 10 | Data Replay | Captures old truthful messages and re-broadcasts them later. |
| **Attacks** | 11 | DoS (Denial of Service) | Spamming the channel to prevent others from communicating. |
| **Attacks** | 12 | DoS Random | Denial of Service using randomized packet intervals. |
| **Attacks** | 13 | DoS Disruptive | Targeted Denial of Service to break specific safety protocols. |
| **Attacks** | 14 | Data Replay Sybil | Replaying data across multiple fake identities. |
| **Attacks** | 15 | Traffic Congestion Sybil | Creating "ghost" traffic to trick navigation systems into rerouting. |
| **Attacks** | 16 | DoS Random Sybil | Large-scale DoS using multiple fake identities. |
| **Attacks** | 17 | DoS Disruptive Sybil | Coordinated disruptive attack using a swarm of fake identities. |
| **Attacks*** | 18 | Extended Attack A | *Likely advanced Sybil or Position spoofing variant.* |
| **Attacks*** | 19 | Extended Attack B | *Likely advanced DoS or Trajectory spoofing variant.* |

*\*Classes 18 and 19 are identified as part of the Extended Attack category based on the numerical sequence and dataset structure.*

In [50]:
# ============================================================
# CELL 1.5 — Dataset Exploration & Sanity Check (Dummy Cell)
# ============================================================
import pandas as pd
import os
from collections import defaultdict

print("--- 🔍 EXTENDED VEREMI DATASET EXPLORATION ---\n")

# 1. Locate the file automatically
fpath = None
if os.path.exists(DATA_DIR):
    for fname in os.listdir(DATA_DIR):
        if fname.endswith('.csv'):
            fpath = os.path.join(DATA_DIR, fname)
            break

if not fpath:
    print(f" Could not find CSV in {DATA_DIR}")
else:
    file_size_gb = os.path.getsize(fpath) / (1024**3)
    print(f"File Found: {os.path.basename(fpath)} ({file_size_gb:.2f} GB)\n")

    # 2. Get Columns & Head (Super fast read of just 5 rows)
    df_sample = pd.read_csv(fpath, nrows=5)
    all_cols = df_sample.columns.tolist()
    
    # These are the exact columns we will extract in Cell 2 for our model
    selected_cols = [
        'sender', 'class', 
        'posx_n', 'posy_n', 'posz_n', 
        'spdx_n', 'spdy_n', 'spdz_n'
    ]

    print(f"[1] TOTAL COLUMNS: {len(all_cols)}")
    print(f"    {all_cols}\n")
    
    print(f"[2] COLUMNS SELECTED FOR PROJECT: {len(selected_cols)}")
    print(f"    Metadata: ['sender', 'class']")
    print(f"    Features: ['posx_n', 'posy_n', 'posz_n', 'spdx_n', 'spdy_n', 'spdz_n']\n")

    print(f"[3] DATA PREVIEW (First 5 Rows):")
    display(df_sample[selected_cols].head())  # Showing only selected for clarity
    print("\n")

    # 4. Memory-Safe Scan for Total Rows & Unique Values
    print("[4] Scanning entire 1.13 GB file for metrics (This will take ~30 seconds)...")
    
    total_rows = 0
    unique_counts = {col: set() for col in all_cols}
    vehicle_row_counts = defaultdict(int) # Tracks rows per physical vehicle
    
    chunk_iter = pd.read_csv(fpath, chunksize=250000)
    for chunk in chunk_iter:
        total_rows += len(chunk)
        for col in all_cols:
            unique_counts[col].update(chunk[col].unique())
            
        # Count rows per sender in this chunk
        for sender_id, count in chunk['sender'].value_counts().items():
            vehicle_row_counts[sender_id] += count

    # 5. Combined Total Rows & Unique Value Counts
    print(f"\n[5] TOTAL ROWS: {total_rows:,}\n")
    print("    UNIQUE VALUES PER COLUMN:")
    for col in all_cols:
        unique_len = len(unique_counts[col])
        flag = " (Selected)" if col in selected_cols else " (Ignored)"
        print(f"    {col:<15} : {unique_len:>9,} unique values  {flag}")
        
    # 6. Rows Per Vehicle Tracker
    print(f"\n[6] ROWS PER VEHICLE (Total Physical Senders: {len(vehicle_row_counts):,}):")
    
    # Sort vehicles from most rows to least rows
    sorted_vehicles = sorted(vehicle_row_counts.items(), key=lambda x: x[1], reverse=True)
    
    MAX_DISPLAY = 50 # Prevents console spam if there are hundreds of cars
    for i, (sender_id, count) in enumerate(sorted_vehicles):
        if i < MAX_DISPLAY:
            print(f"    Sender {sender_id:<12} : {count:>9,} rows")
        elif i == MAX_DISPLAY:
            print(f"    ... and {len(sorted_vehicles) - MAX_DISPLAY} more vehicles.")
            break

    print("\n--- EXPLORATION COMPLETE ---")

--- 🔍 EXTENDED VEREMI DATASET EXPLORATION ---

File Found: mixalldata_clean.csv (1.13 GB)

[1] TOTAL COLUMNS: 30
    ['type', 'sendTime', 'sender', 'senderPseudo', 'messageID', 'class', 'posx', 'posy', 'posz', 'posx_n', 'posy_n', 'posz_n', 'spdx', 'spdy', 'spdz', 'spdx_n', 'spdy_n', 'spdz_n', 'aclx', 'acly', 'aclz', 'aclx_n', 'acly_n', 'aclz_n', 'hedx', 'hedy', 'hedz', 'hedx_n', 'hedy_n', 'hedz_n']

[2] COLUMNS SELECTED FOR PROJECT: 8
    Metadata: ['sender', 'class']
    Features: ['posx_n', 'posy_n', 'posz_n', 'spdx_n', 'spdy_n', 'spdz_n']

[3] DATA PREVIEW (First 5 Rows):


,sender,class,posx_n,posy_n,posz_n,spdx_n,spdy_n,spdz_n
0,130137,0,3.480882,3.473184,0.0,-0.000000,-0.000000,0.0
1,130137,0,3.546261,3.570524,0.0,0.000107,-0.001040,0.0
2,130137,0,3.544045,3.432068,0.0,0.000279,-0.002701,0.0
3,130137,0,3.340080,3.350806,0.0,0.000450,-0.004356,0.0
4,130137,0,3.328872,3.319050,0.0,0.000643,-0.006208,0.0




[4] Scanning entire 1.13 GB file for metrics (This will take ~30 seconds)...

[5] TOTAL ROWS: 3,194,808

    UNIQUE VALUES PER COLUMN:
    type            :         1 unique values   (Ignored)
    sendTime        : 3,194,808 unique values   (Ignored)
    sender          :    24,663 unique values   (Selected)
    senderPseudo    :   118,909 unique values   (Ignored)
    messageID       : 3,194,808 unique values   (Ignored)
    class           :        20 unique values   (Selected)
    posx            : 2,665,316 unique values   (Ignored)
    posy            : 2,665,325 unique values   (Ignored)
    posz            :         1 unique values   (Ignored)
    posx_n          : 2,511,396 unique values   (Selected)
    posy_n          : 2,511,396 unique values   (Selected)
    posz_n          :         1 unique values   (Selected)
    spdx            : 2,541,966 unique values   (Ignored)
    spdy            : 2,541,972 unique values   (Ignored)
    spdz            :         1 unique values 

In [51]:
# ============================================================
# CELL 2 — Data Loading, Models, Client, Server
# ============================================================


class VeReMiDataset(Dataset):
    def __init__(self, sequences, labels):
        self.X = torch.tensor(sequences, dtype=torch.float32)
        self.y = torch.tensor(labels,    dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

def load_veremi(data_dir, window_size, train_split, seed=42):
    import pandas as pd
    import numpy as np
    
    fpath = None
    for fname in os.listdir(data_dir):
        if fname.endswith('.csv'):
            fpath = os.path.join(data_dir, fname)
            break
    if not fpath: raise FileNotFoundError(f"No CSV found in {data_dir}")

    # Physical features (3D Transmitted/Noisy)
    feature_keys = ['posx_n', 'posy_n', 'posz_n', 'spdx_n', 'spdy_n', 'spdz_n']
    label_col = 'class'
    
    print(f"  Streaming Extended Dataset: {os.path.basename(fpath)}")
    
    CHUNK_SIZE = 100000 
    MAX_CLIENTS = 100
    client_data_buffers = {}
    
    chunk_iter = pd.read_csv(fpath, chunksize=CHUNK_SIZE, usecols=['sender', label_col] + feature_keys)
    
    for chunk in chunk_iter:
        for sender_id, group in chunk.groupby('sender'):
            sid_str = str(sender_id)
            if sid_str not in client_data_buffers:
                if len(client_data_buffers) >= MAX_CLIENTS: continue
                client_data_buffers[sid_str] = []
            
            feats = group[feature_keys].values.astype(np.float32)
            labs = group[label_col].values.astype(np.int64)
            client_data_buffers[sid_str].append((feats, labs))
            
        total_rows = sum(sum(len(f) for f, l in v) for v in client_data_buffers.values())
        if total_rows > 4_000_000: break

    all_clients = []
    for sender_id, data_list in client_data_buffers.items():
        full_feats = np.concatenate([d[0] for d in data_list])
        full_labs = np.concatenate([d[1] for d in data_list])
        if len(full_feats) <= window_size: continue
            
        seqs, window_labs = [], []
        for i in range(len(full_feats) - window_size):
            seqs.append(full_feats[i : i + window_size])
            window_labs.append(full_labs[i + window_size])
            
        all_clients.append(VeReMiDataset(np.array(seqs), np.array(window_labs)))

    rng = np.random.RandomState(seed)
    rng.shuffle(all_clients)
    split_idx = int(len(all_clients) * train_split)
    train_clients = all_clients[:split_idx]
    test_clients_raw = all_clients[split_idx:]
    
    test_dataset = VeReMiDataset(
        np.concatenate([c.X.numpy() for c in test_clients_raw]),
        np.concatenate([c.y.numpy() for c in test_clients_raw])
    )
    
    print(f"  Clients — train: {len(train_clients)}, test: {len(test_clients_raw)}")
    return train_clients, test_dataset, len(feature_keys)



In [52]:
# ── Models ───────────────────────────────────────────────────
class BiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_size, num_layers, num_classes, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_size, num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.drop(out[:, -1, :])
        return self.fc(out)


class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_size, num_layers, num_classes, dropout):
        super().__init__()
        self.gru  = nn.GRU(
            input_dim, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.gru(x)
        out = self.drop(out[:, -1, :])
        return self.fc(out)


def build_model(model_type, input_dim, cfg):
    """Factory — returns a freshly initialised model on DEVICE."""
    if model_type == 'bilstm':
        m = BiLSTM(input_dim, cfg['hidden_size'], cfg['num_layers'],
                   cfg['num_classes'], cfg['dropout'])
    elif model_type == 'gru':
        m = GRUModel(input_dim, cfg['hidden_size'], cfg['num_layers'],
                     cfg['num_classes'], cfg['dropout'])
    else:
        raise ValueError(f"Unknown model: {model_type}")
    return m.to(DEVICE)


# ── Churn helper ─────────────────────────────────────────────
def get_churn_p(churn_type, round_num, cfg):
    if churn_type == 'static':
        return cfg['static_churn']
    for (lo, hi), p in cfg['dynamic_schedule'].items():
        if lo <= round_num <= hi:
            return p
    return cfg['static_churn']


# ── Client ───────────────────────────────────────────────────
def client_train(client_id, global_state, dataset, cfg, round_num):
    """
    Local training for one client.
    Returns (delta: state_dict, timestamp: float, difficulty: float)
    """
    # Build a local model and load global weights
    input_dim = dataset[0][0].shape[-1]
    # Infer model type from global_state keys (bilstm has lstm, gru has gru)
    # We pass model_type explicitly via closure — see run_experiment
    local_model = copy.deepcopy(global_state['model'])
    local_model.train()

    loader    = DataLoader(dataset, batch_size=cfg['batch_size'], shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(local_model.parameters(), lr=cfg['lr'])

    total_loss = 0.0
    for _ in range(cfg['local_epochs']):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(local_model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    difficulty = total_loss / (cfg['local_epochs'] * max(len(loader), 1))

    # Compute delta = U_i - G
    global_params = global_state['params']
    delta = {
        k: local_model.state_dict()[k].cpu() - global_params[k].cpu()
        for k in global_params
    }

    # SparseUpdate: keep top sparse_frac of parameters by absolute value
    sparse_frac = cfg['sparse_frac']
    for k in delta:
        flat    = delta[k].abs().flatten()
        if flat.numel() == 0: continue
        thresh  = torch.quantile(flat.float(), 1.0 - sparse_frac)
        mask    = delta[k].abs() >= thresh
        delta[k] = delta[k] * mask

    timestamp = time.time()
    return delta, timestamp, difficulty


# ── Aggregation strategies ────────────────────────────────────
def aggregate(buffer, strategy, global_params, round_num):
    """
    buffer: list of (delta, timestamp, difficulty, submit_round)
    Returns aggregated_delta as state_dict.
    """
    deltas      = [b[0] for b in buffer]
    timestamps  = [b[1] for b in buffer]
    difficulties= [b[2] for b in buffer]
    submit_rounds=[b[3] for b in buffer]

    n = len(deltas)

    # Base weight per strategy
    if strategy == 'noweight':
        weights = np.ones(n)

    elif strategy == 'roundgap':
        staleness = [max(0, round_num - sr) for sr in submit_rounds]
        weights   = np.array([1.0 / (1.0 + s) for s in staleness])

    elif strategy == 'cosinesim':
        # Global direction = mean of all deltas
        def flatten_delta(d):
            return torch.cat([v.flatten().float() for v in d.values()])
        flat_deltas = [flatten_delta(d) for d in deltas]
        global_dir  = torch.stack(flat_deltas).mean(0)
        cos         = nn.CosineSimilarity(dim=0)
        weights     = np.array([
            max(0.0, cos(fd, global_dir).item())
            for fd in flat_deltas
        ])
        weights = np.where(weights == 0, 1e-6, weights)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    # AdaptiveClient: multiply by difficulty
    diff_arr = np.array(difficulties)
    weights  = weights * diff_arr

    # Normalise
    total   = weights.sum()
    weights = weights / total if total > 0 else np.ones(n) / n

    # Weighted aggregation
    agg = {}
    for k in global_params:
        agg[k] = sum(weights[i] * deltas[i][k].float() for i in range(n))
    return agg


# ── Evaluation ───────────────────────────────────────────────
def evaluate(model, test_dataset, cfg):
    model.eval()
    loader = DataLoader(test_dataset, batch_size=cfg['batch_size'] * 2)
    criterion = nn.CrossEntropyLoss()
    all_preds, all_labels, total_loss = [], [], 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits  = model(xb)
            total_loss += criterion(logits, yb).item()
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(yb.cpu().tolist())
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    loss= total_loss / len(loader)
    f1  = f1_score(all_labels, all_preds, average=None,
                   labels=list(range(cfg['num_classes'])), zero_division=0)
    return acc, loss, f1.tolist(), all_preds, all_labels


print("Cell 2.1 complete —  models, client, server, evaluate defined")

Cell 2.1 complete —  models, client, server, evaluate defined


In [53]:
# ============================================================
# CELL 3 — run_experiment() + checkpoint helpers + data load
# ============================================================

def find_checkpoint(key):
    """
    Check TWO locations for a saved result:
      1. /kaggle/working/  — your own previously saved run
      2. /kaggle/input/veremi-results/  — uploaded by a teammate
    Returns (path, source) or (None, None) if not found.
    """
    fname = f"{key}.pt"
    # Check own output first
    own_path = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(own_path):
        return own_path, 'working'
    # Check teammate upload
    upload_path = os.path.join(UPLOAD_DIR, fname)
    if os.path.exists(upload_path):
        return upload_path, 'uploaded'
    return None, None


def save_checkpoint(key, results):
    """Save results dict to /kaggle/working/."""
    path = os.path.join(OUTPUT_DIR, f"{key}.pt")
    torch.save(results, path)
    print(f"  Saved → {path}")
    return path


def run_experiment(model_type, strategy, churn_type, cfg, train_clients, test_dataset, feat_dim, seed=42):
    torch.manual_seed(seed)
    global_model = build_model(model_type, feat_dim, cfg)
    acc_history, loss_history = [], []
    update_buffer = []
    all_nodes = list(range(len(train_clients)))

    print(f"\n[SYSTEM] STARTING AFL SIMULATION")
    print(f"[CONFIG] TOTAL NODES: {len(all_nodes)}  |  GOAL: {cfg['buffer_size']} UPDATES")
    print("=" * 90)

    for rnd in range(1, cfg['num_rounds'] + 1):
        # 1. Check who is online
        p = cfg['static_churn']
        active = [i for i in all_nodes if random.random() > p]
        if not active: active = [random.choice(all_nodes)]
        dropout = [i for i in all_nodes if i not in active]

        print(f"\n[ROUND {rnd:02d}] -----------------------------------------------------------")
        print(f" FLEET STATUS :  Total: {len(all_nodes)}  |  Active: {len(active)}  |  Dropout: {len(dropout)}")
        
        if dropout:
            print(f" OFFLINE      :  Nodes {', '.join(map(str, dropout[:8]))}...")

        global_params = {k: v.clone().cpu() for k, v in global_model.state_dict().items()}
        global_state = {'model': global_model, 'params': global_params}

        # 2. Get data from vehicles (WITH VANET STRAGGLER SIMULATION)
        count = 0
        with ThreadPoolExecutor(max_workers=min(len(active), 8)) as executor:
            futures = {executor.submit(client_train, cid, global_state, train_clients[cid], cfg, rnd): cid for cid in active}
            for future in as_completed(futures):
                try:
                    delta, ts, diff = future.result()
                    
                    # --- NETWORK DELAY SIMULATION ---
                    # 25% chance the vehicle loses signal and the update is delayed
                    if random.random() < 0.25:
                        update_buffer.append((delta, ts, diff, rnd)) 
                    else:
                        # 75% chance it arrives immediately
                        update_buffer.insert(0, (delta, ts, diff, rnd)) 
                    count += 1
                except:
                    pass 

        # 3. Combine updates (TRUE ASYNCHRONOUS FEDBUFF)
        print(f" DATA IN      :  New: {count}  |  Total in Queue: {len(update_buffer)}")

        if len(update_buffer) >= cfg['buffer_size']:
            # Take exactly the buffer size (e.g., 15)
            batch = update_buffer[:cfg['buffer_size']]
            # Leave the delayed stragglers in the queue for future rounds!
            update_buffer = update_buffer[cfg['buffer_size']:] 
            
            # Calculate Age based ONLY on the batch being processed
            age = np.mean([rnd - item[3] for item in batch])
            
            print(f" SERVER       :  Goal met. Processing {len(batch)} updates.")
            print(f" STATS        :  Data Age: {age:.2f} rounds  |  Left in Queue: {len(update_buffer)}")
            
            agg_delta = aggregate(batch, strategy, global_params, rnd)
            new_state = {k: v.cpu() + agg_delta[k] for k, v in global_model.state_dict().items()}
            global_model.load_state_dict({k: v.to(DEVICE) for k, v in new_state.items()})
            
            print(f" UPDATE       :  Global model updated using {strategy.upper()}.")
        else:
            print(f" SERVER       :  Not enough data. Waiting for more nodes.")

        # 4. Check results
        acc, loss, f1, _, _ = evaluate(global_model, test_dataset, cfg)
        acc_history.append(acc)
        loss_history.append(loss)
        
        trend = "📈" if (len(acc_history) < 2 or acc > acc_history[-2]) else "📉"
        print(f" RESULTS      :  Accuracy: {acc:.4f} {trend}  |  Loss: {loss:.4f}")

    # 5. FINAL COMPREHENSIVE EVALUATION (To fix the NameError)
    print("\n" + "="*90)
    print("🏁 [SIMULATION FINISHED] GENERATING FINAL REPORT...")
    final_acc, final_loss, final_f1, preds, labels = evaluate(global_model, test_dataset, cfg)

    return {
        'model_type': model_type, 'strategy': strategy, 'churn_type': churn_type,
        'acc_history': acc_history, 'loss_history': loss_history,
        'final_acc': final_acc, 
        'final_loss': final_loss, 
        'final_f1': final_f1,
        'report': classification_report(
            labels, 
            preds, 
            labels=list(range(cfg['num_classes'])), 
            target_names=[f'Class_{i}' for i in range(cfg['num_classes'])], 
            zero_division=0
        ),
        'config': cfg
    }



print("Loading VeReMi dataset...")
TRAIN_CLIENTS, TEST_DATASET, FEAT_DIM = load_veremi(DATA_DIR, CFG['window_size'], CFG['train_split'], seed=SEED)
print(f"Cell 3 complete — {len(TRAIN_CLIENTS)} clients ready | feat_dim={FEAT_DIM}")


Loading VeReMi dataset...
  Streaming Extended Dataset: mixalldata_clean.csv
  Clients — train: 80, test: 20
Cell 3 complete — 80 clients ready | feat_dim=6


---
## Runs — BiLSTM
Cells 4–9: BiLSTM with all 3 strategies × 2 churn settings

In [ ]:
# ============================================================
# CELL 4 — BiLSTM + NoWeighting + Static Churn
# ============================================================

KEY = 'bilstm_noweight_static'
ckpt_path, ckpt_src = find_checkpoint(KEY)

if ckpt_path is not None:
    print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
    print(f"       Loading from: {ckpt_path}")
    R04 = torch.load(ckpt_path)
    print(f"       Final Acc: {R04['final_acc']:.4f}")
else:
    print(f"[RUN ] Training: {KEY}")
    R04 = run_experiment(
        model_type='bilstm', strategy='noweight', churn_type='static',
        cfg=CFG, train_clients=TRAIN_CLIENTS,
        test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
    )
    save_checkpoint(KEY, R04)
    print(f"[DONE] Final Acc: {R04['final_acc']:.4f}")
    print(R04['report'])

In [ ]:
# # ============================================================
# # CELL 5 — BiLSTM + NoWeighting + Dynamic Churn
# # ============================================================

# KEY = 'bilstm_noweight_dynamic'
# ckpt_path, ckpt_src = find_checkpoint(KEY)

# if ckpt_path is not None:
#     print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
#     print(f"       Loading from: {ckpt_path}")
#     R05 = torch.load(ckpt_path)
#     print(f"       Final Acc: {R05['final_acc']:.4f}")
# else:
#     print(f"[RUN ] Training: {KEY}")
#     R05 = run_experiment(
#         model_type='bilstm', strategy='noweight', churn_type='dynamic',
#         cfg=CFG, train_clients=TRAIN_CLIENTS,
#         test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
#     )
#     save_checkpoint(KEY, R05)
#     print(f"[DONE] Final Acc: {R05['final_acc']:.4f}")
#     print(R05['report'])

In [ ]:
# ============================================================
# CELL 6 — BiLSTM + RoundGap + Static Churn
# ============================================================

KEY = 'bilstm_roundgap_static'
ckpt_path, ckpt_src = find_checkpoint(KEY)

if ckpt_path is not None:
    print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
    print(f"       Loading from: {ckpt_path}")
    R06 = torch.load(ckpt_path)
    print(f"       Final Acc: {R06['final_acc']:.4f}")
else:
    print(f"[RUN ] Training: {KEY}")
    R06 = run_experiment(
        model_type='bilstm', strategy='roundgap', churn_type='static',
        cfg=CFG, train_clients=TRAIN_CLIENTS,
        test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
    )
    save_checkpoint(KEY, R06)
    print(f"[DONE] Final Acc: {R06['final_acc']:.4f}")
    print(R06['report'])

In [ ]:
# # ============================================================
# # CELL 7 — BiLSTM + RoundGap + Dynamic Churn
# # ============================================================

# KEY = 'bilstm_roundgap_dynamic'
# ckpt_path, ckpt_src = find_checkpoint(KEY)

# if ckpt_path is not None:
#     print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
#     print(f"       Loading from: {ckpt_path}")
#     R07 = torch.load(ckpt_path)
#     print(f"       Final Acc: {R07['final_acc']:.4f}")
# else:
#     print(f"[RUN ] Training: {KEY}")
#     R07 = run_experiment(
#         model_type='bilstm', strategy='roundgap', churn_type='dynamic',
#         cfg=CFG, train_clients=TRAIN_CLIENTS,
#         test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
#     )
#     save_checkpoint(KEY, R07)
#     print(f"[DONE] Final Acc: {R07['final_acc']:.4f}")
#     print(R07['report'])

In [ ]:
# ============================================================
# CELL 8 — BiLSTM + CosineSim + Static Churn
# ============================================================

KEY = 'bilstm_cosinesim_static'
ckpt_path, ckpt_src = find_checkpoint(KEY)

if ckpt_path is not None:
    print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
    print(f"       Loading from: {ckpt_path}")
    R08 = torch.load(ckpt_path)
    print(f"       Final Acc: {R08['final_acc']:.4f}")
else:
    print(f"[RUN ] Training: {KEY}")
    R08 = run_experiment(
        model_type='bilstm', strategy='cosinesim', churn_type='static',
        cfg=CFG, train_clients=TRAIN_CLIENTS,
        test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
    )
    save_checkpoint(KEY, R08)
    print(f"[DONE] Final Acc: {R08['final_acc']:.4f}")
    print(R08['report'])

In [ ]:
# # ============================================================
# # CELL 9 — BiLSTM + CosineSim + Dynamic Churn
# # ============================================================

# KEY = 'bilstm_cosinesim_dynamic'
# ckpt_path, ckpt_src = find_checkpoint(KEY)

# if ckpt_path is not None:
#     print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
#     print(f"       Loading from: {ckpt_path}")
#     R09 = torch.load(ckpt_path)
#     print(f"       Final Acc: {R09['final_acc']:.4f}")
# else:
#     print(f"[RUN ] Training: {KEY}")
#     R09 = run_experiment(
#         model_type='bilstm', strategy='cosinesim', churn_type='dynamic',
#         cfg=CFG, train_clients=TRAIN_CLIENTS,
#         test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
#     )
#     save_checkpoint(KEY, R09)
#     print(f"[DONE] Final Acc: {R09['final_acc']:.4f}")
#     print(R09['report'])

---
## Runs — GRU
Cells 10–15: GRU with all 3 strategies × 2 churn settings

In [ ]:
# ============================================================
# CELL 10 — GRU + NoWeighting + Static Churn
# ============================================================

KEY = 'gru_noweight_static'
ckpt_path, ckpt_src = find_checkpoint(KEY)

if ckpt_path is not None:
    print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
    print(f"       Loading from: {ckpt_path}")
    R10 = torch.load(ckpt_path)
    print(f"       Final Acc: {R10['final_acc']:.4f}")
else:
    print(f"[RUN ] Training: {KEY}")
    R10 = run_experiment(
        model_type='gru', strategy='noweight', churn_type='static',
        cfg=CFG, train_clients=TRAIN_CLIENTS,
        test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
    )
    save_checkpoint(KEY, R10)
    print(f"[DONE] Final Acc: {R10['final_acc']:.4f}")
    print(R10['report'])

In [ ]:
# # ============================================================
# # CELL 11 — GRU + NoWeighting + Dynamic Churn
# # ============================================================

# KEY = 'gru_noweight_dynamic'
# ckpt_path, ckpt_src = find_checkpoint(KEY)

# if ckpt_path is not None:
#     print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
#     print(f"       Loading from: {ckpt_path}")
#     R11 = torch.load(ckpt_path)
#     print(f"       Final Acc: {R11['final_acc']:.4f}")
# else:
#     print(f"[RUN ] Training: {KEY}")
#     R11 = run_experiment(
#         model_type='gru', strategy='noweight', churn_type='dynamic',
#         cfg=CFG, train_clients=TRAIN_CLIENTS,
#         test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
#     )
#     save_checkpoint(KEY, R11)
#     print(f"[DONE] Final Acc: {R11['final_acc']:.4f}")
#     print(R11['report'])

In [ ]:
# ============================================================
# CELL 12 — GRU + RoundGap + Static Churn
# ============================================================

KEY = 'gru_roundgap_static'
ckpt_path, ckpt_src = find_checkpoint(KEY)

if ckpt_path is not None:
    print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
    print(f"       Loading from: {ckpt_path}")
    R12 = torch.load(ckpt_path)
    print(f"       Final Acc: {R12['final_acc']:.4f}")
else:
    print(f"[RUN ] Training: {KEY}")
    R12 = run_experiment(
        model_type='gru', strategy='roundgap', churn_type='static',
        cfg=CFG, train_clients=TRAIN_CLIENTS,
        test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
    )
    save_checkpoint(KEY, R12)
    print(f"[DONE] Final Acc: {R12['final_acc']:.4f}")
    print(R12['report'])

In [ ]:
# # ============================================================
# # CELL 13 — GRU + RoundGap + Dynamic Churn
# # ============================================================

# KEY = 'gru_roundgap_dynamic'
# ckpt_path, ckpt_src = find_checkpoint(KEY)

# if ckpt_path is not None:
#     print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
#     print(f"       Loading from: {ckpt_path}")
#     R13 = torch.load(ckpt_path)
#     print(f"       Final Acc: {R13['final_acc']:.4f}")
# else:
#     print(f"[RUN ] Training: {KEY}")
#     R13 = run_experiment(
#         model_type='gru', strategy='roundgap', churn_type='dynamic',
#         cfg=CFG, train_clients=TRAIN_CLIENTS,
#         test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
#     )
#     save_checkpoint(KEY, R13)
#     print(f"[DONE] Final Acc: {R13['final_acc']:.4f}")
#     print(R13['report'])

In [ ]:
# ============================================================
# CELL 14 — GRU + CosineSim + Static Churn
# ============================================================

KEY = 'gru_cosinesim_static'
ckpt_path, ckpt_src = find_checkpoint(KEY)

if ckpt_path is not None:
    print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
    print(f"       Loading from: {ckpt_path}")
    R14 = torch.load(ckpt_path)
    print(f"       Final Acc: {R14['final_acc']:.4f}")
else:
    print(f"[RUN ] Training: {KEY}")
    R14 = run_experiment(
        model_type='gru', strategy='cosinesim', churn_type='static',
        cfg=CFG, train_clients=TRAIN_CLIENTS,
        test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
    )
    save_checkpoint(KEY, R14)
    print(f"[DONE] Final Acc: {R14['final_acc']:.4f}")
    print(R14['report'])

In [ ]:
# # ============================================================
# # CELL 15 — GRU + CosineSim + Dynamic Churn
# # ============================================================

# KEY = 'gru_cosinesim_dynamic'
# ckpt_path, ckpt_src = find_checkpoint(KEY)

# if ckpt_path is not None:
#     print(f"[SKIP] '{KEY}' already exists ({ckpt_src}).")
#     print(f"       Loading from: {ckpt_path}")
#     R15 = torch.load(ckpt_path)
#     print(f"       Final Acc: {R15['final_acc']:.4f}")
# else:
#     print(f"[RUN ] Training: {KEY}")
#     R15 = run_experiment(
#         model_type='gru', strategy='cosinesim', churn_type='dynamic',
#         cfg=CFG, train_clients=TRAIN_CLIENTS,
#         test_dataset=TEST_DATASET, feat_dim=FEAT_DIM, seed=SEED
#     )
#     save_checkpoint(KEY, R15)
#     print(f"[DONE] Final Acc: {R15['final_acc']:.4f}")
#     print(R15['report'])

---
## Analysis
Cells 16–18: Load all saved results and generate plots. Can be run any time — even if only some runs are complete.

In [ ]:
# ============================================================
# CELL 16 — Load all results + Accuracy vs Rounds plot
# ============================================================

# ALL_KEYS = [
#     'bilstm_noweight_static',  'bilstm_noweight_dynamic',
#     'bilstm_roundgap_static',  'bilstm_roundgap_dynamic',
#     'bilstm_cosinesim_static', 'bilstm_cosinesim_dynamic',
#     'gru_noweight_static',     'gru_noweight_dynamic',
#     'gru_roundgap_static',     'gru_roundgap_dynamic',
#     'gru_cosinesim_static',    'gru_cosinesim_dynamic',
# ]
ALL_KEYS = [
    'bilstm_noweight_static',
    'bilstm_roundgap_static',
    'bilstm_cosinesim_static',
    'gru_noweight_static',
    'gru_roundgap_static',
    'gru_cosinesim_static',
]


ALL_RESULTS = {}
for key in ALL_KEYS:
    path, src = find_checkpoint(key)
    if path:
        ALL_RESULTS[key] = torch.load(path)
        print(f"  Loaded [{src:8s}] {key:40s} acc={ALL_RESULTS[key]['final_acc']:.4f}")
    else:
        print(f"  Missing            {key}")

print(f"\n{len(ALL_RESULTS)}/12 runs available for plotting")

# ── Accuracy vs Rounds ────────────────────────────────────────
COLORS   = {'noweight': '#1f77b4', 'roundgap': '#ff7f0e', 'cosinesim': '#2ca02c'}
LSTYLES  = {'static': '-', 'dynamic': '--'}
MARKERS  = {'bilstm': 'o', 'gru': 's'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle('Accuracy vs Communication Rounds', fontsize=14, fontweight='bold')

for ax, model_type in zip(axes, ['bilstm', 'gru']):
    for key, res in ALL_RESULTS.items():
        if res['model_type'] != model_type: continue
        strategy   = res['strategy']
        churn_type = res['churn_type']
        label = f"{strategy} / {churn_type}"
        ax.plot(
            range(1, len(res['acc_history']) + 1),
            res['acc_history'],
            color    = COLORS[strategy],
            linestyle= LSTYLES[churn_type],
            marker   = MARKERS[model_type],
            markevery= 10,
            markersize=5,
            linewidth= 1.8,
            label    = label
        )
    ax.set_title(model_type.upper(), fontsize=12)
    ax.set_xlabel('Round')
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(1, CFG['num_rounds'])

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/accuracy_vs_rounds.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: accuracy_vs_rounds.png")

In [ ]:
# ============================================================
# CELL 17 — F1 Score per Class Heatmaps
# ============================================================

CLASS_NAMES = ['Normal', 'Attack_1', 'Attack_2', 'Attack_3', 'Attack_4', 'Attack_5']

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('F1 Score per Class — Final Round', fontsize=14, fontweight='bold')

for ax, model_type in zip(axes, ['bilstm', 'gru']):
    rows, row_labels = [], []
    for key in ALL_KEYS:
        if key not in ALL_RESULTS: continue
        res = ALL_RESULTS[key]
        if res['model_type'] != model_type: continue
        row_labels.append(f"{res['strategy']} / {res['churn_type']}")
        rows.append(res['final_f1'])

    if not rows:
        ax.set_title(f"{model_type.upper()} — no data")
        continue

    df_f1 = pd.DataFrame(rows, index=row_labels, columns=CLASS_NAMES)
    sns.heatmap(
        df_f1, ax=ax, annot=True, fmt='.3f',
        cmap='YlOrRd', vmin=0, vmax=1,
        linewidths=0.5, cbar_kws={'label': 'F1 Score'}
    )
    ax.set_title(model_type.upper(), fontsize=12)
    ax.set_xlabel('Class')
    ax.set_ylabel('Strategy / Churn')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/f1_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: f1_heatmap.png")

In [ ]:
# ============================================================
# CELL 18 — Final Accuracy Comparison Table
# ============================================================

rows = []
for key in ALL_KEYS:
    if key not in ALL_RESULTS:
        rows.append({'Key': key, 'Model': '—', 'Strategy': '—',
                     'Churn': '—', 'Final Acc': 'pending',
                     'Final Loss': 'pending', 'Macro F1': 'pending'})
        continue
    res = ALL_RESULTS[key]
    macro_f1 = np.mean(res['final_f1'])
    rows.append({
        'Key'       : key,
        'Model'     : res['model_type'].upper(),
        'Strategy'  : res['strategy'],
        'Churn'     : res['churn_type'],
        'Final Acc' : f"{res['final_acc']:.4f}",
        'Final Loss': f"{res['final_loss']:.4f}",
        'Macro F1'  : f"{macro_f1:.4f}",
    })

df_summary = pd.DataFrame(rows).drop(columns=['Key'])
print("\n===== AFL-VeReMi-Bench — Final Results Summary =====")
print(df_summary.to_string(index=False))

# Save as CSV
df_summary.to_csv(f"{OUTPUT_DIR}/results_summary.csv", index=False)
print(f"\nSaved: results_summary.csv")

# Best run
done = df_summary[df_summary['Final Acc'] != 'pending'].copy()
if not done.empty:
    done['_acc'] = done['Final Acc'].astype(float)
    best = done.loc[done['_acc'].idxmax()]
    print(f"\nBest run: {best['Model']} | {best['Strategy']} | {best['Churn']} "
          f"→ Acc={best['Final Acc']}  MacroF1={best['Macro F1']}")